In [110]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as ply
%matplotlib inline

In [111]:
words = open('names.txt', 'r').read().splitlines()
len(words)

32033

In [112]:
# Build mapping system
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [113]:
# Build the dataset
block_size = 3
X, Y = [], [] # Context and label
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
        print("".join(itos[i] for i in context), '--->', itos[ix])
X = torch.tensor(X)
Y = torch.tensor(Y)
print(f"\nX shape: {X.shape}")
print(f"Y shape: {Y.shape}")

emma
..e ---> e
.em ---> m
emm ---> m
mma ---> a
ma. ---> .
olivia
..o ---> o
.ol ---> l
oli ---> i
liv ---> v
ivi ---> i
via ---> a
ia. ---> .
ava
..a ---> a
.av ---> v
ava ---> a
va. ---> .
isabella
..i ---> i
.is ---> s
isa ---> a
sab ---> b
abe ---> e
bel ---> l
ell ---> l
lla ---> a
la. ---> .
sophia
..s ---> s
.so ---> o
sop ---> p
oph ---> h
phi ---> i
hia ---> a
ia. ---> .

X shape: torch.Size([32, 3])
Y shape: torch.Size([32])


Check Bengio et. Al, 2003 paper and Andej Karpathy's walkthrough to better clarify architecture. Key points:
- The model looks at 4 words, and aims to predict the 5th.
- Each word is given a 30 dimension vector embedding.
- Every (17000) word's embedding is stored inside a 17000 by 30 lookup matrix, C, which is used to map each of the 4 previous words to their embeddings, before being fed into the neural network to get predictions for the 5th word. 

For our experiment, we are going to make letter embeddings 2 dimensions, since there are only 27 possibilities compared to 17000 words in the paper.

In [114]:
C = torch.randn((27, 2)) # Lookup matrix contains a 2 dimension embedding for each of the 27 characters

In [115]:
C[X] # Lookup the embedding for each character in X
# The output is the embedding (2d) for each character in X (32 inputs x 3 characters x 2 dimensions)

tensor([[[-0.3148,  0.4877],
         [-0.3148,  0.4877],
         [-0.3148,  0.4877]],

        [[-0.3148,  0.4877],
         [-0.3148,  0.4877],
         [-0.6250, -1.0044]],

        [[-0.3148,  0.4877],
         [-0.6250, -1.0044],
         [-0.2372,  2.0049]],

        [[-0.6250, -1.0044],
         [-0.2372,  2.0049],
         [-0.2372,  2.0049]],

        [[-0.2372,  2.0049],
         [-0.2372,  2.0049],
         [ 0.0620, -0.1603]],

        [[-0.3148,  0.4877],
         [-0.3148,  0.4877],
         [-0.3148,  0.4877]],

        [[-0.3148,  0.4877],
         [-0.3148,  0.4877],
         [-0.4404,  0.2556]],

        [[-0.3148,  0.4877],
         [-0.4404,  0.2556],
         [ 0.5240,  0.5988]],

        [[-0.4404,  0.2556],
         [ 0.5240,  0.5988],
         [ 0.7131, -0.9109]],

        [[ 0.5240,  0.5988],
         [ 0.7131, -0.9109],
         [-1.3851, -2.3727]],

        [[ 0.7131, -0.9109],
         [-1.3851, -2.3727],
         [ 0.7131, -0.9109]],

        [[-1.3851, -2

In [116]:
C[X].shape

torch.Size([32, 3, 2])

In [117]:
emb = C[X]

Hidden layer

In [118]:
W1 = torch.randn((6, 100)) # 3x2 weights per neuron, 100 neurons
b1 = torch.randn(100) # 100 bias terms

In [119]:
# What we'd like to do:
# emb @ W1 + b1
# This won't work since emb is 32x3x2 and W1 is 6x100
# We must reshape emb from 32x3x2 into 32x6
# We can use the .view() method to reshape the tensor

In [120]:
W1.shape

torch.Size([6, 100])

In [121]:
h = emb.view((-1), 6) @ W1 + b1

OR

In [122]:
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)

In [123]:
# torch.cat(torch.unbind(emb, 1), 1)

Between emb.view and torch.cat, we generally opt for emb.view, since its much more storage and compute efficient.

In [124]:
h = torch.tanh(h)
h

tensor([[ 0.4693, -0.9810,  0.8856,  ..., -0.2102, -0.8302, -0.9674],
        [-0.9172, -0.9342,  0.6112,  ..., -0.0096,  0.9880, -0.8842],
        [ 0.9923, -0.9989,  0.9640,  ...,  0.8109, -0.9999, -0.9942],
        ...,
        [ 0.9795,  0.7458,  0.8597,  ...,  0.9167, -0.9366, -0.4570],
        [ 0.9363,  0.9998, -0.3937,  ..., -0.6310,  0.9527,  0.9996],
        [ 0.9808,  0.9694, -0.4742,  ..., -0.2520,  0.3384,  0.9997]])

Output layer

In [126]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [127]:
logits = h @ W2 + b2

In [130]:
# Treat the logits as log values that need to be exponentiated
counts = logits.exp()

In [131]:
probs = counts / counts.sum(1, keepdims=True)

In [137]:
loss = -probs[torch.arange(32), Y].log().mean()
loss

tensor(17.1183)

In [135]:
probs

tensor([[3.6797e-03, 4.4458e-13, 3.1485e-08, 6.3839e-10, 1.7673e-09, 1.0683e-10,
         1.7177e-06, 3.0705e-12, 2.6325e-06, 8.2961e-07, 9.1174e-09, 9.0627e-10,
         9.9607e-15, 2.6461e-16, 5.5450e-09, 1.7974e-08, 1.1973e-07, 1.2667e-15,
         2.4024e-08, 9.9630e-01, 4.9516e-12, 9.5413e-08, 1.6751e-05, 1.3463e-09,
         6.2880e-11, 1.3244e-08, 6.6357e-11],
        [9.4713e-03, 8.4915e-11, 1.2639e-05, 5.6424e-10, 1.2622e-07, 1.2139e-05,
         4.1178e-04, 1.2407e-13, 5.2644e-08, 1.5410e-06, 6.0161e-09, 1.0184e-08,
         4.2715e-06, 2.3555e-12, 5.9427e-04, 2.5032e-03, 1.2983e-02, 8.0718e-09,
         1.1744e-04, 9.7305e-01, 6.8645e-11, 5.8320e-06, 3.5127e-06, 7.1111e-04,
         2.7582e-05, 8.9041e-05, 4.6976e-09],
        [2.8899e-07, 1.2956e-18, 1.2117e-14, 1.0640e-11, 1.2094e-15, 7.8732e-20,
         4.6680e-16, 1.4794e-18, 1.8326e-10, 8.8675e-16, 9.9584e-16, 7.1873e-17,
         5.9743e-22, 8.5796e-20, 4.9915e-17, 8.0034e-18, 9.3164e-16, 9.6904e-23,
         8.4103e-

All together:

In [138]:
# Generating random weights for the entire network
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [140]:
sum(p.nelement() for p in parameters)

3481

In [141]:
# Forward pass
emb = C[X]
h = torch.tanh(emb.view((-1), 6) @ W1 + b1)
logits = h @ W2 + b2
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)
loss = -probs[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)